# Inversión promocional DEX · todas las compañías · cortes comparables

Reporte enfocado en **inversión promocional** y **CcC** (clientes con compra). Universo: todos los cod_negocio (filtro «Negocio»: CMP = CP1/CP2, el resto B2B); se excluyen las compañías (DEX) y los clientes sin venta en los últimos 3 meses (`MESES_SIN_VENTA`). Solo se extraen las categorías **Salsas y Conservas** (`CATEGORIAS`).

Compara el **mismo corte** (días 1 al N de cada mes) en los 3 meses anteriores y el mismo mes del año anterior; cada día que se ejecuta, el corte avanza solo al último día con datos.

- **KPIs**: inversión promocional y CcC. Sin filtro de categoría ni familia, el CcC cuenta los clientes que compraron **Salsas o Conservas** (`CATS_CCC`; un cliente que compró las dos cuenta una vez); con una categoría o familia elegida, los que compraron en ella; con un tipo de inversión elegido, los que recibieron esa inversión.
- **Filtros**: tipo de inversión, categoría, familia, negocio y boca.
- **Detalle («Abrir por»)**: tipo de inversión, categoría, marca, negocio y boca.
- **Familia**: cada `cod_familia` toma su último nombre (`des_familia` de la línea más reciente) para homologar nombres.

Corre el notebook **dos veces con el mismo corte**: `FECHA_BASE = "facturacion"` y `FECHA_BASE = "pedido"`. Cada corrida genera su archivo de datos (`data_facturacion.js` / `data_pedido.js`); el selector «Fecha» del reporte cambia entre ellos sin recargar.

Genera `outputs/inversion_promocional_<etiqueta>.html` (usa `plantilla_evolucion.html`; se publica como `index.html` junto a los dos `data_*.js`) y `outputs/clientes_con_compra_<etiqueta>.csv`. Correr de arriba a abajo.


## 1. Parámetros

In [118]:
from pathlib import Path
import datetime as dt

# ── Fecha de corte y ventanas comparables ───────────────────────────────────
FECHA_CORTE = None                   # None = último día con datos cargados (se actualiza solo cada día). O una fecha fija: dt.date(2026, 11, 2)
MODO_CORTE = "mes"                   # "mes":       cada mes acumula del día 1 al día del corte (si el corte es el 3: del 1 al 3 en todos los meses)
                                     # "acumulado": se acumulan N días (N = día del mes del corte) desde el 1.º mismo día de la semana con que empieza el mes del corte
                                     #              (si el mes empieza en domingo, cada mes de comparación arranca en su primer domingo)
                                     # "ventana":   últimos N_DIAS días que terminan en el n-ésimo mismo día de la semana de cada mes
N_DIAS = 3                           # solo MODO_CORTE = "ventana": largo de cada ventana
N_MESES_ATRAS = 3                    # meses anteriores a comparar
ANIOS_ATRAS = 1                      # años atrás (mismo mes del año anterior)

# ── Fecha base: qué fecha arma las ventanas ──────────────────────────────────
FECHA_BASE = "facturacion"           # "facturacion": fec_documento de dex_cliente_detalle
                                     # "pedido":      fecha de pedido de Odoo (odoo_mk_dex_cliente_venta, cruce por id_pedido_venta; sin pedido en Odoo se usa la de facturación)
                                     # Correr una vez con cada valor y el mismo corte: el HTML tiene un selector «Fecha» que cambia entre data_facturacion.js y data_pedido.js sin recargar
MESES_POST_PEDIDO = 2                # solo "pedido": meses siguientes a cada ventana donde se busca la factura del pedido (un pedido de octubre puede facturarse en noviembre)
assert FECHA_BASE in ("facturacion", "pedido")

# ── Universo ────────────────────────────────────────────────────────────────
PROJECT_ID = "acpe-prod-dlk-shrd-gld"

# Boca de salida (la de cr_segmentacion; si el cliente no está, la de dex_cliente). No se filtra: BO (incluye AB y OT), PM, MM y MA se quedan; el resto es B2B
NEGOCIOS = None                      # None = todos los cod_negocio. Ej: ["CP1", "CP2"] para limitar
CMP_NEGOCIOS = ["CP1", "CP2"]        # cod_negocio que cuentan como «CMP»; cualquier otro es «B2B» (filtro «Negocio» del reporte)
MESES_SIN_VENTA = 3                  # se quitan las compañías (DEX) y los clientes sin venta en los últimos N meses hasta el corte (None = no filtrar)
COMPANIAS_ID = None                  # id_compania_origen. None = todas las DEX. Ej: ["D011", "D965"] para limitar
OFICINAS = None                      # None = todas. Ej: ["CM-SUR / CENTRO"]
CATS_CCC = ["Salsas", "Conservas"]   # KPI CcC sin filtro de categoría: clientes que compraron alguna de estas (una vez cada cliente). Deben escribirse igual que des_categoria
CATEGORIAS = CATS_CCC                # categorías que se extraen: solo Salsas y Conservas (None = todas)
# Días con descuento para el calendario: {categoría: {"año-mes": {"3": [días 3 %], "6": [días 6 %]}}}. Vacío = el calendario solo marca domingos
DESCUENTO_DIAS = {}
USAR_CACHE = True                   # True = reutiliza la extracción guardada en data/ sin consultar BigQuery

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "notebooks" / "data"
OUT_DIR = BASE_DIR / "outputs"
PLANTILLA_HTML = BASE_DIR / "plantilla_evolucion.html"
LOGO = BASE_DIR / "assets" / "logo_final_texto_derecha.png"
SUF = "" if FECHA_BASE == "facturacion" else "_pedido"      # sufijo de los archivos de la base «pedido»
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(BASE_DIR)

c:\Users\VCACERESC\OneDrive - Grupo Alicorp\Documentos\Promociones\gasto_promocional


## 2. Ventanas comparables
Modo `acumulado`: el mes del corte empieza en un día de la semana (p. ej. domingo); cada mes de comparación arranca en su primer mismo día de la semana y acumula los mismos días hasta el corte (si el mes se acaba antes, hasta su último día). Modo `ventana`: termina en el mismo ordinal y día de la semana del corte y retrocede `N_DIAS` días.

In [119]:
import pandas as pd
import pandas_gbq

MESES_ES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
DIAS_ES = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
DIAS_ES_LARGO = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]


def sumar_meses(d, k):
    y, m = divmod(d.year * 12 + d.month - 1 + k, 12)
    return dt.date(y, m + 1, 1)


def corte_equivalente(ini_mes, dow, n):
    """n-ésimo día de la semana `dow` (0=lun) del mes que empieza en ini_mes; si no existe, el último."""
    c = ini_mes + dt.timedelta(days=(dow - ini_mes.weekday()) % 7 + 7 * (n - 1))
    return c if c.month == ini_mes.month else c - dt.timedelta(days=7)


def txt_fecha(d):
    return f"{DIAS_ES[d.weekday()]} {d.day}-{MESES_ES[d.month - 1]}"


def ultimo_dia_con_datos():
    """Último día cargado en dex_cliente_detalle (la data llega hasta el día anterior a hoy)."""
    desde = sumar_meses(dt.date.today().replace(day=1), -1)      # literal para pruning de particiones
    filtro_neg = f"AND cod_negocio IN ({', '.join(repr(x) for x in NEGOCIOS)})" if NEGOCIOS else ""
    f = pandas_gbq.read_gbq(
        f"""SELECT CAST(MAX(fec_documento) AS STRING) AS f
            FROM `acpe-prod-dlk-shrd-gld.gld_cliente.dex_cliente_detalle`
            WHERE periodo >= DATE '{desde}' {filtro_neg}""",
        project_id=PROJECT_ID, dialect="standard", progress_bar_type=None)["f"][0]
    return dt.date.fromisoformat(f)


hoy = FECHA_CORTE or ultimo_dia_con_datos()          # día de corte
dow, n = hoy.weekday(), (hoy.day - 1) // 7 + 1
mes_hoy = hoy.replace(day=1)


def fin_de_mes(m):
    return sumar_meses(m, 1) - dt.timedelta(days=1)


if MODO_CORTE in ("mes", "acumulado"):
    # El mes del corte empieza en `dow1` (p. ej. domingo). En cada mes de comparación se arranca en su primer
    # mismo día de la semana y se acumulan los mismos `hoy.day` días: todas las ventanas tienen los mismos días
    # de la semana y crecen juntas día a día. Si se acaba el mes, la ventana se corta en su último día.
    dow1 = mes_hoy.weekday()

    def ventana_mes(m):
        ini = m if MODO_CORTE == "mes" else m + dt.timedelta(days=(dow1 - m.weekday()) % 7)
        return ini, min(ini + dt.timedelta(days=hoy.day - 1), fin_de_mes(m))

    que = (f"Acumulado del 1 al {hoy.day} de cada mes" if MODO_CORTE == "mes"
           else f"Acumulado de {hoy.day} días desde el 1.º {DIAS_ES_LARGO[dow1]}")
    cortes = [("actual", *ventana_mes(mes_hoy), que)]
    cortes += [("mes", *ventana_mes(sumar_meses(mes_hoy, -k)), que) for k in range(1, N_MESES_ATRAS + 1)]
    cortes += [("anio", *ventana_mes(sumar_meses(mes_hoy, -12 * a)), que + " · año anterior") for a in range(1, ANIOS_ATRAS + 1)]
else:
    fines = [("actual", hoy, "Corte actual")]
    fines += [("mes", corte_equivalente(sumar_meses(mes_hoy, -k), dow, n), f"{n}.º {DIAS_ES_LARGO[dow]} del mes")
              for k in range(1, N_MESES_ATRAS + 1)]
    fines += [("anio", hoy - dt.timedelta(days=364 * a), f"Hace {52 * a} semanas (mismo día de la semana)")
              for a in range(1, ANIOS_ATRAS + 1)]
    cortes = [(t, f - dt.timedelta(days=N_DIAS - 1), f, q) for t, f, q in fines]

VENTANAS = []
for orden, (tipo, ini, fin, que) in enumerate(sorted(cortes, key=lambda c: c[2])):
    VENTANAS.append(dict(etiqueta=f"{MESES_ES[fin.month - 1].capitalize()}-{fin:%y}", orden=orden, tipo=tipo,
                         ini=ini, fin=fin, rango=f"{txt_fecha(ini)} → {txt_fecha(fin)}", queEs=que,
                         dias=(fin - ini).days + 1))

SEG_PERIODO = sumar_meses(mes_hoy, -2)      # segmentación m-2 respecto al mes del corte (foto única para todas las ventanas)
ETIQUETA = f"corte_{hoy:%Y%m%d}_" + {"mes": "mes", "acumulado": "acum"}.get(MODO_CORTE, f"{N_DIAS}d")

# Meses completos para el calendario diario (el mes del corte llega hasta `hoy`). Orden: meses anteriores, mes del corte, año anterior
ORD_TIPO = {"mes": 0, "actual": 1, "anio": 2}
VENT_DIA = []
for i, v in enumerate(sorted(VENTANAS, key=lambda v: (ORD_TIPO[v["tipo"]], v["fin"]))):
    m0 = v["fin"].replace(day=1)
    VENT_DIA.append(dict(etiqueta=v["etiqueta"], orden=i, tipo=v["tipo"], ini=m0, fin=min(fin_de_mes(m0), hoy)))

print(f"Corte: {DIAS_ES_LARGO[dow]} {hoy} | modo: {MODO_CORTE} | segmentación: {SEG_PERIODO}")
pd.DataFrame(VENTANAS)[["orden", "etiqueta", "tipo", "ini", "fin", "rango", "queEs"]]

Corte: miércoles 2026-11-18 | modo: mes | segmentación: 2026-09-01


,orden,etiqueta,tipo,ini,fin,rango,queEs
0,0,Nov-25,anio,2025-11-01,2025-11-18,sáb 1-nov → mar 18-nov,Acumulado del 1 al 18 de cada mes · año anterior
1,1,Ago-26,mes,2026-08-01,2026-08-18,sáb 1-ago → mar 18-ago,Acumulado del 1 al 18 de cada mes
2,2,Sep-26,mes,2026-09-01,2026-09-18,mar 1-sep → vie 18-sep,Acumulado del 1 al 18 de cada mes
3,3,Oct-26,mes,2026-10-01,2026-10-18,jue 1-oct → dom 18-oct,Acumulado del 1 al 18 de cada mes
4,4,Nov-26,actual,2026-11-01,2026-11-18,dom 1-nov → mié 18-nov,Acumulado del 1 al 18 de cada mes


## 3. Extracción (BigQuery)
Consultas sobre la misma base (línea de pedido × material, con el cliente):

- **celdas**: por ventana, tipo de inversión, negocio y boca, y por nivel de producto (todas · categoría · categoría+familia · categoría+marca · categoría+familia+marca), con **inversión**, monto neto y CcC. Tipo y negocio van como filas «específico» y «todos», para que los clientes distintos no se sumen dos veces.
- **dias**: lo mismo por día (sin marca), para el calendario.
- **clientes**: un registro por cliente y ventana (se guarda como CSV).
- **familias**: códigos de familia con más de un nombre en el periodo (control de la homologación).

**Inversión** = `-SUM(IF(es_inv, mnt_bruto, 0))`, con `es_inv` = toda línea TPR/TDD (incluye las «Sin descuento»). «Sin descuento» no sale en el filtro de tipo de inversión, ni como fila del detalle por tipo, pero su inversión siempre se suma al total, sea cual sea el tipo elegido (en el CcC por tipo no cuenta).
**CcC** = clientes distintos con líneas de venta (sin TPR/TDD); en las filas de un tipo de inversión, clientes distintos que recibieron esa inversión.


In [120]:
import numpy as np
import pandas_gbq

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)

DIMS = ["tipo", "negocio", "boca"]       # tipo = tipo de inversión · negocio = CMP (CP1/CP2) o B2B · boca: una por cliente (sin canal ni segmento)
SIN_INV = "Sin inversión"                # «tipo» de las líneas que no son inversión (las ventas, no TPR/TDD): no sale en el reporte


def lista(xs):
    return ", ".join(f"'{x}'" for x in xs)


def meses_cubiertos(vent=None):
    ms = set()
    for v in (vent or VENTANAS):          # solo los meses de cada ventana (no los intermedios)
        d = v["ini"].replace(day=1)
        while d <= v["fin"]:
            ms.add(d.isoformat()); d = sumar_meses(d, 1)
    return sorted(ms)


ODOO_PEDIDOS = "`acpe-prod-dlk-shrd-gld.gld_cliente.odoo_mk_dex_cliente_venta`"


def meses_factura(vent=None):
    """Particiones de dex_cliente_detalle (periodo = mes de facturación). En base «pedido» se agregan los meses siguientes a cada ventana."""
    ms = set(meses_cubiertos(vent))
    if FECHA_BASE == "pedido":
        for m in list(ms):
            for k in range(1, MESES_POST_PEDIDO + 1):
                x = sumar_meses(dt.date.fromisoformat(m), k)
                if x <= mes_hoy:
                    ms.add(x.isoformat())
    return sorted(ms)


def sql_base(vent=None):
    vent = vent or VENTANAS
    ventanas = ",\n    ".join(
        f"STRUCT('{v['etiqueta']}' AS etiqueta, {v['orden']} AS orden, DATE '{v['ini']}' AS ini, DATE '{v['fin']}' AS fin)"
        for v in vent)
    filtro_ofi = f"AND a.des_oficina_venta IN ({lista(OFICINAS)})" if OFICINAS else ""
    filtro_cat = f"AND a.des_categoria IN ({lista(CATEGORIAS)})" if CATEGORIAS else ""
    filtro_comp = f"AND a.id_compania_origen IN ({lista(COMPANIAS_ID)})" if COMPANIAS_ID else ""
    filtro_neg = f"AND a.cod_negocio IN ({lista(NEGOCIOS)})" if NEGOCIOS else ""
    if MESES_SIN_VENTA:      # compañías (DEX) y clientes con venta (sin TPR/TDD) en los últimos N meses hasta el corte; el resto no entra
        m0 = sumar_meses(mes_hoy, -MESES_SIN_VENTA)
        desde_act = m0 + dt.timedelta(days=min(hoy.day, fin_de_mes(m0).day))
        meses_act, d_act = [], m0
        while d_act <= mes_hoy:
            meses_act.append(d_act.isoformat()); d_act = sumar_meses(d_act, 1)
        cte_activos = f""", ventas_recientes AS (   -- ventas de los últimos {MESES_SIN_VENTA} meses hasta el corte (todos los negocios)
  SELECT DISTINCT a.id_interlocutor_origen, a.id_compania_origen
  FROM `acpe-prod-dlk-shrd-gld.gld_cliente.dex_cliente_detalle` a
  WHERE a.periodo IN ({lista(meses_act)})
    AND a.fec_documento BETWEEN DATE '{desde_act}' AND DATE '{hoy}'
    AND COALESCE(a.cod_clase_posicion, '') NOT IN ('TPR', 'TDD')
    {filtro_ofi}
    {filtro_comp}
    {filtro_neg}
)
, dex_activas AS (SELECT DISTINCT id_compania_origen FROM ventas_recientes)
, clientes_activos AS (SELECT DISTINCT id_interlocutor_origen FROM ventas_recientes)"""
        filtro_activos = "AND a.id_compania_origen IN (SELECT id_compania_origen FROM dex_activas) AND a.id_interlocutor_origen IN (SELECT id_interlocutor_origen FROM clientes_activos)"
    else:
        cte_activos, filtro_activos = "", ""
    if FECHA_BASE == "pedido":
        desde = sumar_meses(dt.date.fromisoformat(meses_cubiertos(vent)[0]), -2)
        cte_pedidos = f""", pedidos_odoo AS (   -- fecha de pedido por id_pedido (MIN: una fecha por pedido, así el cruce no duplica líneas)
  SELECT CAST(id_pedido_venta AS STRING) AS id_pedido, MIN(DATE(fec_pedido_venta)) AS fec_pedido
  FROM {ODOO_PEDIDOS}
  WHERE periodo BETWEEN DATE '{desde}' AND DATE '{mes_hoy}'
  GROUP BY 1
)"""
        join_odoo = "LEFT JOIN pedidos_odoo o ON CAST(a.id_pedido_venta AS STRING) = o.id_pedido"
        fecha_expr = "COALESCE(o.fec_pedido, a.fec_documento)"      # sin pedido en Odoo la línea se queda con su fecha de facturación
    else:
        cte_pedidos, join_odoo, fecha_expr = "", "", "a.fec_documento"
    return f"""
WITH ventanas AS (
  SELECT * FROM UNNEST([
    {ventanas}
  ])
)
, promociones AS (
  SELECT cod_promocion, ANY_VALUE(des_promocion) AS des_promocion
  FROM `acpe-prod-dlk-shrd-gld.delivery_comercial_dex.cr_promocion_detalle`
  WHERE periodo >= '2025-01-01'
  GROUP BY cod_promocion
)
{cte_pedidos}
{cte_activos}
, segmento AS (   -- boca: foto única de segmentación (m-2 del mes del corte) para todas las ventanas
  SELECT cod_interlocutor, cod_boca_salida
  FROM `acpe-prod-dlk-shrd-gld.delivery_cliente_gtm.cr_segmentacion`
  WHERE periodo = '{SEG_PERIODO}'
)
, boca_dex AS (   -- boca del maestro de clientes, una fila por cliente
  SELECT id_interlocutor_origen, MAX(NULLIF(UPPER(TRIM(cod_boca_salida)), '')) AS cod_boca_salida
  FROM `acpe-prod-dlk-shrd-gld.gld_cliente.dex_cliente`
  GROUP BY id_interlocutor_origen
)
, lineas AS (
  SELECT
    v.etiqueta, v.orden,
    {fecha_expr} AS fecha,
    CASE WHEN UPPER(TRIM(COALESCE(NULLIF(TRIM(s.cod_boca_salida), ''), b.cod_boca_salida))) IN ('BO', 'AB', 'OT') THEN 'BO'
         WHEN UPPER(TRIM(COALESCE(NULLIF(TRIM(s.cod_boca_salida), ''), b.cod_boca_salida))) IN ('PM', 'MM', 'MA')
           THEN UPPER(TRIM(COALESCE(NULLIF(TRIM(s.cod_boca_salida), ''), b.cod_boca_salida)))
         ELSE 'B2B' END AS boca,
    CASE WHEN a.cod_negocio IN ({lista(CMP_NEGOCIOS)}) THEN 'CMP' ELSE 'B2B' END AS negocio,
    COALESCE(a.des_categoria, 'Sin dato') AS categoria,
    COALESCE(a.des_categoria, 'Sin dato') IN ({lista(CATS_CCC)}) AS en_ccc,      -- categorías del KPI CcC (Salsas y Conservas)
    a.cod_familia,
    NULLIF(TRIM(a.des_familia), '') AS des_familia,
    COALESCE(NULLIF(TRIM(a.des_marca_material), ''), 'Sin dato') AS marca,
    a.id_interlocutor_origen, a.id_pedido_venta,
    COALESCE(a.cod_clase_posicion IN ('TPR', 'TDD'), FALSE) AS es_promo,
    a.mnt_bruto,
    CASE
      WHEN a.cod_promocion_s4 = 'TPM-007' THEN 'Combo DEX'
      WHEN p.des_promocion LIKE '%B1%' THEN 'Descuento digital'
      WHEN a.cod_promocion_s4 IN ('TPM-001','TPM-002','TPM-003','TPM-004','TPM-005',
                                  'TPM-006','TPM-008','TPM-009')
           THEN 'Promociones de Piso'
      WHEN REGEXP_CONTAINS(p.des_promocion, r'(D-PR|DPR|-PR-)') THEN 'Promociones de Piso'
      WHEN a.cod_promocion_s4 IS NULL THEN 'Sin descuento'
      ELSE 'Promociones de Piso'   -- OTROS se incluye en piso
    END AS tipo_promocion
  FROM `acpe-prod-dlk-shrd-gld.gld_cliente.dex_cliente_detalle` a
  {join_odoo}
  JOIN ventanas v
    ON {fecha_expr} BETWEEN v.ini AND v.fin
  LEFT JOIN boca_dex b ON a.id_interlocutor_origen = b.id_interlocutor_origen
  LEFT JOIN segmento s ON a.id_interlocutor_origen = s.cod_interlocutor
  LEFT JOIN promociones p ON p.cod_promocion = a.id_promocion_origen
  WHERE a.periodo IN ({lista(meses_factura(vent))})   -- literales para pruning
    {filtro_ofi}
    {filtro_cat}
    {filtro_comp}
    {filtro_neg}
    {filtro_activos}
)
, familias AS (   -- homologación: cada cod_familia se queda con su último nombre (el de la línea más reciente)
  SELECT cod_familia, ARRAY_AGG(des_familia ORDER BY fecha DESC LIMIT 1)[OFFSET(0)] AS familia
  FROM lineas
  WHERE cod_familia IS NOT NULL AND des_familia IS NOT NULL
  GROUP BY cod_familia
)
, base AS (
  SELECT l.*,
    COALESCE(f.familia, l.des_familia, 'Sin dato') AS familia,
    l.es_promo AS es_inv,      -- inversión = todas las líneas TPR/TDD (incluye las «Sin descuento»)
    IF(l.es_promo, l.tipo_promocion, '{SIN_INV}') AS tipo
  FROM lineas l
  LEFT JOIN familias f ON l.cod_familia = f.cod_familia
)
"""


NIV_PROD = ["", ", categoria", ", categoria, familia", ", categoria, marca", ", categoria, familia, marca"]
NIV_DIA = ["", ", categoria", ", categoria, familia"]          # el calendario no se abre por marca


def _sets(pref, niveles=NIV_PROD):
    """Grupos de agregación: tipo y negocio van como filas «específico» y como filas «todos» (NULL), para que los clientes distintos no se dupliquen
    cuando un cliente recibe más de un tipo de inversión o compra en más de un negocio; cada combinación con cada nivel de producto."""
    out = []
    for tipo in ("tipo, ", ""):
        for neg in ("negocio, ", ""):
            for prod in niveles:
                out.append(f"  ({pref}, {tipo}{neg}boca{prod})")
    return ",\n".join(out)


METRICAS = """
  -SUM(IF(es_inv, mnt_bruto, 0)) AS inversion,                                              -- inversión promocional
  SUM(mnt_bruto) AS monto,                                                                       -- monto neto (control)
  COUNT(DISTINCT IF(es_promo, NULL, id_interlocutor_origen)) AS clientes,                         -- CcC: clientes con líneas de venta
  COUNT(DISTINCT IF(NOT es_promo AND en_ccc, id_interlocutor_origen, NULL)) AS clientes_ccc,      -- CcC de Salsas y Conservas (una vez por cliente)
  COUNT(DISTINCT IF(es_inv, id_interlocutor_origen, NULL)) AS clientes_inv,                       -- clientes que recibieron inversión (filas por tipo)
  COUNT(DISTINCT IF(es_inv AND en_ccc, id_interlocutor_origen, NULL)) AS clientes_ccc_inv"""

Q_CELDAS = """
SELECT etiqueta, orden, tipo, negocio, boca, categoria, familia, marca,""" + METRICAS + """,
  COUNT(DISTINCT IF(es_promo, NULL, fecha)) AS dias_venta,
  CAST(MAX(fecha) AS STRING) AS fecha_max
FROM base
GROUP BY GROUPING SETS (
""" + _sets("etiqueta, orden") + """,
  (etiqueta, orden)                                                          -- total de la ventana (solo cobertura)
)
"""

Q_DIAS = """
SELECT etiqueta, EXTRACT(DAY FROM fecha) AS dia, tipo, negocio, boca, categoria, familia,""" + METRICAS + """
FROM base
GROUP BY GROUPING SETS (
""" + _sets("etiqueta, fecha", NIV_DIA) + """
)
"""

Q_CLIENTES = """
SELECT etiqueta, CAST(id_interlocutor_origen AS STRING) AS cod_cliente, boca,
  STRING_AGG(DISTINCT negocio ORDER BY negocio) AS negocios,
  SUM(mnt_bruto) AS monto,
  -SUM(IF(es_inv, mnt_bruto, 0)) AS inversion,
  LOGICAL_OR(NOT es_promo AND en_ccc) AS compra_salsas_conservas
FROM base
GROUP BY 1, 2, 3
HAVING COUNTIF(NOT es_promo) > 0
"""

Q_FAMILIAS = """
SELECT CAST(cod_familia AS STRING) AS cod_familia, ANY_VALUE(familia) AS familia_homologada,
  STRING_AGG(DISTINCT des_familia, ' | ') AS nombres, COUNT(DISTINCT des_familia) AS n_nombres
FROM base
WHERE cod_familia IS NOT NULL
GROUP BY 1
HAVING COUNT(DISTINCT des_familia) > 1
"""


def leer(nombre, cuerpo, vent=None):
    f = DATA_DIR / f"inv_{nombre}_v1_{ETIQUETA}{SUF}.parquet"
    if USAR_CACHE and f.exists():
        print(f"cache: {f.name}")
        return pd.read_parquet(f)
    d = pandas_gbq.read_gbq(sql_base(vent) + cuerpo, project_id=PROJECT_ID, dialect="standard", progress_bar_type=None)
    d.to_parquet(f, index=False)
    print(f"BigQuery -> {f.name} ({len(d):,} filas)")
    return d


df_celdas = leer("celdas", Q_CELDAS)
df_dias = leer("dias", Q_DIAS, VENT_DIA)
df_clientes = leer("clientes", Q_CLIENTES)
df_familias = leer("familias", Q_FAMILIAS)

CLI_COLS = ["clientes", "clientes_ccc", "clientes_inv", "clientes_ccc_inv"]
for c in ["inversion", "monto", "dias_venta"] + CLI_COLS:
    df_celdas[c] = df_celdas[c].astype(float)
df_celdas["orden"] = df_celdas["orden"].astype(int)
for c in ["inversion", "monto", "dia"] + CLI_COLS:
    df_dias[c] = df_dias[c].astype(float)
for c in ["monto", "inversion"]:
    df_clientes[c] = df_clientes[c].astype(float)

archivo_clientes = OUT_DIR / f"clientes_con_compra_{ETIQUETA}{SUF}.csv"
df_clientes.to_csv(archivo_clientes, index=False, encoding="utf-8-sig")
print(f"✓ {len(df_celdas):,} celdas | {len(df_dias):,} filas diarias | {len(df_clientes):,} clientes-ventana -> {archivo_clientes.name}")
print(f"familias con más de un nombre (homologadas al último): {len(df_familias):,}")
df_raw = df_celdas          # alias: la celda de inspección `df_raw` sigue funcionando


cache: inv_celdas_v1_corte_20261118_mes.parquet
cache: inv_dias_v1_corte_20261118_mes.parquet
cache: inv_clientes_v1_corte_20261118_mes.parquet
cache: inv_familias_v1_corte_20261118_mes.parquet
✓ 11,062 celdas | 109,345 filas diarias | 511,045 clientes-ventana -> clientes_con_compra_corte_20261118_mes.csv
familias con más de un nombre (homologadas al último): 16


## 4. Validación
Cobertura de fechas por ventana; que las aperturas (tipo de inversión, negocio, categoría, familia, marca) sumen al total; que las categorías del CcC existan; y la lista de familias homologadas.


In [121]:
df_raw

,etiqueta,orden,tipo,negocio,boca,categoria,familia,marca,inversion,monto,clientes,clientes_ccc,clientes_inv,clientes_ccc_inv,dias_venta,fecha_max
0,Nov-25,0,Combo DEX,None,PM,Salsas,None,None,33436.22,-33436.22,0.0,0.0,4814.0,4814.0,0.0,2025-11-18
1,Nov-25,0,None,CMP,MA,None,None,None,211616.20,3340296.34,1901.0,1901.0,1855.0,1855.0,15.0,2025-11-18
2,Nov-25,0,None,B2B,BO,Salsas,None,Alpesa,23.76,60530.43,2669.0,2669.0,15.0,15.0,15.0,2025-11-18
3,Nov-25,0,Promociones de Piso,None,BO,Salsas,None,None,42866.39,-42866.39,0.0,0.0,23745.0,23745.0,0.0,2025-11-18
4,Nov-25,0,None,CMP,MA,Salsas,None,Alacena,178767.27,2407192.59,1746.0,1746.0,1698.0,1698.0,15.0,2025-11-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11057,Nov-26,4,Promociones de Piso,None,B2B,Conservas,Primor Atun,Primor,895.20,-895.20,0.0,0.0,1177.0,1177.0,0.0,2026-11-18
11058,Nov-25,0,Promociones de Piso,None,MM,Salsas,Alacena Mayonesa Regular,None,1013.67,-1013.67,0.0,0.0,86.0,86.0,0.0,2025-11-18
11059,Oct-26,3,Sin descuento,B2B,BO,Salsas,Alpesa Ketchup FS Granel,None,-0.43,0.43,0.0,0.0,1.0,1.0,0.0,2026-10-17
11060,Ago-26,1,Sin descuento,None,PM,Salsas,None,Alpesa,-43.37,43.37,0.0,0.0,5.0,5.0,0.0,2026-08-18


In [122]:
vent = df_celdas[df_celdas["boca"].isna()].set_index("etiqueta")          # total de la ventana
cel = df_celdas[df_celdas["boca"].notna()]
sin_tn = cel["tipo"].isna() & cel["negocio"].isna()                       # filas «todos los tipos» y «todos los negocios»
niv = {"todo": cel["categoria"].isna(),
       "cat": cel["categoria"].notna() & cel["familia"].isna() & cel["marca"].isna(),
       "fam": cel["familia"].notna() & cel["marca"].isna(),
       "marca": cel["marca"].notna() & cel["familia"].isna(),
       "fam_marca": cel["familia"].notna() & cel["marca"].notna()}
ADIT = ["inversion", "monto"]
todo = cel[sin_tn & niv["todo"]].groupby("etiqueta")[ADIT + ["clientes", "clientes_ccc"]].sum()

resumen = pd.DataFrame(VENTANAS).set_index("etiqueta")[["tipo", "ini", "fin"]].join(
    vent[["fecha_max", "inversion", "monto", "clientes", "clientes_ccc", "dias_venta"]])
print(resumen.to_string())

# 1) el corte actual debe llegar hasta su último día
for e, r in resumen[resumen["tipo"] == "actual"].iterrows():
    if pd.isna(r["fecha_max"]) or r["fecha_max"] < r["fin"].isoformat():
        print(f"⚠ {e}: datos hasta {r['fecha_max']}, el corte es el {r['fin']} (ventana incompleta)")

# 2) las bocas (todos los tipos, negocios y categorías) suman al total de la ventana, clientes incluidos (un cliente = una boca)
d = (todo - vent[ADIT + ["clientes", "clientes_ccc"]]).abs().max()
print("Σ bocas vs total ventana             | dif máx:", {k: round(v, 4) for k, v in d.items()})

# 3) los tipos de inversión suman la inversión total
por_tipo = cel[cel["tipo"].notna() & (cel["tipo"] != SIN_INV) & cel["negocio"].isna() & niv["todo"]]
print("Σ tipos de inversión vs total        | dif máx inversión:", round((por_tipo.groupby("etiqueta")["inversion"].sum() - todo["inversion"]).abs().max(), 4))
print(por_tipo.pivot_table(index="tipo", columns="etiqueta", values="inversion", aggfunc="sum").round(0).to_string())

# 4) negocios, categorías, familias y marcas suman igual en inversión y monto
for nombre, m in [("negocios", cel["tipo"].isna() & cel["negocio"].notna() & niv["todo"]),
                  ("categorías", sin_tn & niv["cat"]), ("familias", sin_tn & niv["fam"]),
                  ("marcas", sin_tn & niv["marca"]), ("familia × marca", sin_tn & niv["fam_marca"])]:
    d = (cel[m].groupby("etiqueta")[ADIT].sum() - todo[ADIT]).abs().max()
    print(f"Σ {nombre:<16} vs total          | dif máx:", {k: round(v, 4) for k, v in d.items()})

# 5) categorías del CcC: deben existir con ese nombre; el CcC conjunto no puede pasar la suma de las dos
cats_ok = set(cel["categoria"].dropna())
falta = [c for c in CATS_CCC if c not in cats_ok]
assert not falta, f"CATS_CCC: no encuentro {falta} en des_categoria. Categorías: {sorted(cats_ok)}"
cc_cat = cel[sin_tn & niv["cat"] & cel["categoria"].isin(CATS_CCC)].groupby("etiqueta")["clientes"].sum()
print(pd.DataFrame({"CcC Salsas+Conservas (únicos)": todo["clientes_ccc"], "Σ CcC por categoría": cc_cat,
                    "compran ambas": cc_cat - todo["clientes_ccc"]}).to_string())

# 6) el calendario diario (días 1..corte de cada mes) debe cuadrar con la ventana del mismo mes
if MODO_CORTE == "mes":
    dd = df_dias[df_dias["tipo"].isna() & df_dias["negocio"].isna() & df_dias["categoria"].isna()]
    for v in VENTANAS:
        s = dd[(dd["etiqueta"] == v["etiqueta"]) & (dd["dia"] <= v["fin"].day)][ADIT].sum()
        dif = (s - todo.loc[v["etiqueta"], ADIT]).abs()
        print(f"calendario vs ventana {v['etiqueta']} | dif inversión {dif['inversion']:,.2f} · monto {dif['monto']:,.2f}")
        assert dif.max() < 1, f"{v['etiqueta']}: el calendario diario no cuadra con la ventana"

# 7) cada cliente tiene una sola boca por ventana (condición para sumar clientes entre bocas)
n = df_clientes.groupby(["etiqueta", "cod_cliente"]).size()
print(f"clientes-ventana duplicados: {(n > 1).sum()}")

# 8) familias homologadas: código, nombre que se usa y nombres que tuvo en el periodo
df_familias.sort_values("cod_familia")


            tipo         ini         fin   fecha_max  inversion        monto  clientes  clientes_ccc  dias_venta
etiqueta                                                                                                        
Nov-25      anio  2025-11-01  2025-11-18  2025-11-18  629139.75  16423245.49   83134.0       83134.0        16.0
Ago-26       mes  2026-08-01  2026-08-18  2026-08-18  850717.01  20494350.04  103179.0      103179.0        17.0
Sep-26       mes  2026-09-01  2026-09-18  2026-09-18  978078.72  23715709.01  110461.0      110461.0        17.0
Oct-26       mes  2026-10-01  2026-10-18  2026-10-17  785756.29  20519141.97  107360.0      107360.0        15.0
Nov-26    actual  2026-11-01  2026-11-18  2026-11-18  753925.44  21865415.72  106911.0      106911.0        16.0
Σ bocas vs total ventana             | dif máx: {'inversion': 0.0, 'monto': 0.0, 'clientes': 0.0, 'clientes_ccc': 0.0}
Σ tipos de inversión vs total        | dif máx inversión: 0.0
etiqueta               Ago-2

,cod_familia,familia_homologada,nombres,n_nombres
1,127,Alacena Mostaza Regular,Alacena Mostaza Regular | Alacena Mostaza,2
14,128,Alacena Mostaza FS Sachet,Alacena Mostaza FS Sachet | Alacena Mostaza FS,2
11,131,Alacena Huancaina Regular,Alacena Huancaina Regular | Alacena Huancaina,2
12,132,Alacena Tari Regular,Alacena Tari | Alacena Tari Regular,2
7,133,Alacena Uchucuta Regular,Alacena Uchucuta Regular | Alacena Uchucuta,2
6,134,Alacena Mayonesa Regular,Alacena Mayonesa | Alacena Mayonesa Regular,2
8,135,Alacena Mayonesa FS Granel,Alacena Mayonesa FS Granel | Alacena Mayonesa FS,2
10,136,Alacena Mayonesa Light Regular,Alacena Mayonesa Light Regular | Alacena Mayon...,2
0,137,Alacena Ketchup Regular,Alacena Ketchup Regular | Alacena Ketchup,2
2,138,Alacena Ketchup FS Sachet,Alacena Ketchup FS Sachet | Alacena Ketchup FS,2


## 5. Reporte HTML
Embebe los resultados en `plantilla_evolucion.html` y guarda el reporte en `outputs/`. KPIs de inversión promocional y CcC; filtros de tipo de inversión, categoría, familia, negocio y boca (cada uno con las opciones que existen según los demás); selector de fecha base (facturación / pedido); calendario con cuadro semanal (S1–S4) de la métrica elegida; detalle abierto por tipo de inversión, categoría, marca, negocio o boca, con Δ.


In [123]:
import base64
import json

def logo_b64():
    """Logo de assets recortado a su contenido (sin el margen blanco) para la pastilla del encabezado."""
    if not LOGO.exists():
        print(f"sin logo ({LOGO.name} no existe): se usa el texto")
        return None
    try:
        from io import BytesIO
        from PIL import Image, ImageChops
        im = Image.open(LOGO).convert("RGB")
        dif = ImageChops.difference(im, Image.new("RGB", im.size, im.getpixel((0, 0)))).convert("L")
        caja = dif.point(lambda x: 255 if x > 12 else 0).getbbox()
        im = im.convert("RGBA"); im.putalpha(dif.point(lambda x: min(255, x * 10)))      # fondo transparente: el logo se ve limpio sobre la cabecera blanca
        if caja:
            im = im.crop(caja)
        im.thumbnail((700, 170))
        buf = BytesIO(); im.save(buf, "PNG", optimize=True)
        datos = buf.getvalue()
    except ImportError:
        datos = LOGO.read_bytes()
    return "data:image/png;base64," + base64.b64encode(datos).decode()


BANNER = BASE_DIR / "assets" / "banner_Alicorp.png"       # logo de Alicorp que va arriba del reporte


def banner_b64():
    if not BANNER.exists():
        print(f"sin banner ({BANNER.name} no existe): el reporte sale sin el logo de Alicorp")
        return None
    return "data:image/png;base64," + base64.b64encode(BANNER.read_bytes()).decode()


DIM_LABEL = {"tipo": "Tipo de inversión", "negocio": "Negocio", "boca": "Boca"}
TIPOS_ORDEN = ["Promociones de Piso", "Descuento digital", "Combo DEX"]


def preparar(frame):
    """Quita las filas del tipo «Sin inversión» y, en las filas de un tipo de inversión, CcC = clientes que recibieron esa inversión."""
    f = frame[frame["tipo"] != SIN_INV].copy()          # NaN (= todos los tipos) se queda
    es_t = f["tipo"].notna()
    f.loc[es_t, "clientes"] = f.loc[es_t, "clientes_inv"]
    f.loc[es_t, "clientes_ccc"] = f.loc[es_t, "clientes_ccc_inv"]
    return f


cel = preparar(df_celdas[df_celdas["boca"].notna()])        # incluye las filas «todas las categorías» (categoría vacía = -1 en el HTML)
dd = preparar(df_dias)
vent = df_celdas[df_celdas["boca"].isna()].set_index("etiqueta")

cats = sorted(cel["categoria"].dropna().unique())
valores = {k: sorted(set(cel[k].dropna().unique()) | set(dd[k].dropna().unique())) for k in DIMS}
valores["tipo"] = [t for t in TIPOS_ORDEN if t in valores["tipo"]] + [t for t in valores["tipo"] if t not in TIPOS_ORDEN]
ix = {k: {v: i for i, v in enumerate(valores[k])} for k in DIMS}
cat_ix = {c: i for i, c in enumerate(cats)}
w_ix = {v["etiqueta"]: i for i, v in enumerate(VENTANAS)}


def pares(col):
    """familias / marcas, cada una con su categoría (si un nombre está en varias categorías se distingue como «nombre (categoría)»)."""
    pr = cel[cel[col].notna()][["categoria", col]].drop_duplicates().sort_values([col, "categoria"]).reset_index(drop=True)
    pr["label"] = np.where(pr[col].duplicated(keep=False), pr[col] + " (" + pr["categoria"] + ")", pr[col])
    return pr, {f"{c}\x1f{v}": i for i, (c, v) in enumerate(zip(pr["categoria"], pr[col]))}


fa_pares, fa_ix = pares("familia")
mk_pares, mk_ix = pares("marca")


def ids(frame, col, idx):
    return (frame["categoria"].astype(str) + "\x1f" + frame[col].astype(str)).map(idx).fillna(-1).astype(int)


perfiles, perfil_id = [], {}          # perfil = (tipo | -1 = todos, negocio | -1 = todos, boca)


def perfil_ids(frame):
    arr = np.column_stack([frame[k].map(ix[k]).fillna(-1).astype(int) for k in DIMS])
    out = np.empty(len(arr), dtype=int)
    for j, row in enumerate(map(tuple, arr.tolist())):
        if row not in perfil_id:
            perfil_id[row] = len(perfiles)
            perfiles.append(list(row))
        out[j] = perfil_id[row]
    return out


def r2(s):
    return np.round(s.to_numpy(dtype=float), 2).tolist()


def ints(s):
    return s.fillna(0).astype(int).tolist()


# celdas: inversión y CcC por ventana × perfil × nivel de producto (-1 = todas)
celdas = dict(w=cel["etiqueta"].map(w_ix).astype(int).tolist(), p=perfil_ids(cel).tolist(),
              c=cel["categoria"].map(cat_ix).fillna(-1).astype(int).tolist(), fa=ids(cel, "familia", fa_ix).tolist(), mk=ids(cel, "marca", mk_ix).tolist(),
              inv=r2(cel["inversion"]), cli=ints(cel["clientes"]), cc=ints(cel["clientes_ccc"]))

# por día (calendario): mismo perfil, categoría y familia
mes_ix = {v["etiqueta"]: i for i, v in enumerate(VENT_DIA)}
dias = dict(m=dd["etiqueta"].map(mes_ix).astype(int).tolist(), d=dd["dia"].astype(int).tolist(), p=perfil_ids(dd).tolist(),
            c=dd["categoria"].map(cat_ix).fillna(-1).astype(int).tolist(), fa=ids(dd, "familia", fa_ix).tolist(),
            inv=r2(dd["inversion"]), cli=ints(dd["clientes"]), cc=ints(dd["clientes_ccc"]))
meses_js = []
for v in VENT_DIA:
    k = f"{v['ini']:%Y-%m}"
    marca = {c: dict(d3=DESCUENTO_DIAS.get(c, {}).get(k, {}).get("3", []), d6=DESCUENTO_DIAS.get(c, {}).get(k, {}).get("6", [])) for c in cats}
    meses_js.append(dict(etiqueta=v["etiqueta"], tipo=v["tipo"], anio=v["ini"].year, mes=v["ini"].month,
                         ndias=fin_de_mes(v["ini"]).day, hasta=v["fin"].day, marca=marca))

ventanas_js = [dict(etiqueta=v["etiqueta"], tipo=v["tipo"], rango=v["rango"], queEs=v["queEs"], dias=v["dias"],
                    fin=v["fin"].isoformat(), fecha_max=vent["fecha_max"].get(v["etiqueta"]),
                    diasVenta=int(vent["dias_venta"].get(v["etiqueta"], 0))) for v in VENTANAS]

if MODO_CORTE in ("mes", "acumulado"):
    TITULO = f"Evolución acumulada · al {hoy.day}-{MESES_ES[hoy.month - 1]}-{hoy.year}"
else:
    TITULO = f"Evolución con cortes comparables · {DIAS_ES_LARGO[dow]} {hoy.day}-{MESES_ES[hoy.month - 1]}-{hoy.year}"
SUBTITULO = "Inversión promocional DEX"          # título grande del encabezado

payload = dict(
    meta=dict(
        titulo=TITULO,
        subtitulo=SUBTITULO,
        corteDia=hoy.day,
        fecha_base=FECHA_BASE,
        corte=hoy.isoformat(),
    ),
    ventanas=ventanas_js,
    dims=[dict(key=k, label=DIM_LABEL[k], valores=valores[k]) for k in DIMS],
    cats=cats,
    ccc=CATS_CCC,
    familias=fa_pares["label"].tolist(),
    fam_cat=[cat_ix[c] for c in fa_pares["categoria"]],
    marcas=mk_pares["label"].tolist(),
    marca_cat=[cat_ix[c] for c in mk_pares["categoria"]],
    perfiles=perfiles,
    celdas=celdas,
    meses=meses_js,
    dias=dias,
)
txt = json.dumps(payload, ensure_ascii=False, separators=(",", ":"), allow_nan=False).replace("</", "<\\/")
# los datos van en un archivo por fecha base (data_facturacion.js / data_pedido.js) y el HTML es una cáscara liviana que carga el que se elija
archivo_datos = OUT_DIR / f"data_{FECHA_BASE}.js"
archivo_datos.write_text(f'window.__D = window.__D || {{}}; window.__D["{FECHA_BASE}"] = ' + txt + ";", encoding="utf-8")
cascara = dict(meta=dict(titulo=TITULO, subtitulo=SUBTITULO, corte=hoy.isoformat(), logo=logo_b64(), banner=banner_b64(),
                         archivos=dict(facturacion="data_facturacion.js", pedido="data_pedido.js")))
plantilla = PLANTILLA_HTML.read_text(encoding="utf-8")
assert "/*__DATA__*/null" in plantilla, f"{PLANTILLA_HTML.name}: no encuentro el marcador /*__DATA__*/null (¿el editor lo reformateó?)"
salida = OUT_DIR / f"inversion_promocional_{ETIQUETA}.html"
salida.write_text(plantilla.replace("/*__DATA__*/null", json.dumps(cascara, ensure_ascii=False, separators=(",", ":"), allow_nan=False).replace("</", "<\\/")), encoding="utf-8")
mb = archivo_datos.stat().st_size / 1e6
print(f"✓ {archivo_datos.name} | {mb:.2f} MB | {len(celdas['w']):,} celdas · {len(dias['d']):,} días · {len(perfiles):,} perfiles · {len(fa_pares):,} familias · {len(mk_pares):,} marcas")
print(f"✓ {salida.name} | {salida.stat().st_size / 1e6:.2f} MB (cáscara) — publícalo como index.html junto a data_facturacion.js y data_pedido.js")
if mb > 90:
    print("⚠ pasa de ~90 MB (GitHub rechaza archivos de más de 100 MB)")


✓ data_facturacion.js | 2.23 MB | 8,576 celdas · 82,047 días · 64 perfiles · 23 familias · 5 marcas
✓ inversion_promocional_corte_20261118_mes.html | 0.19 MB (cáscara) — publícalo como index.html junto a data_facturacion.js y data_pedido.js
